# Notebook 04: Reducción de Dimensionalidad (PCA)
## Proyecto Integrador — Minería de Datos I

**Objetivo:** Aplicar PCA sobre las variables numéricas relevantes, documentando las variables utilizadas, el escalamiento aplicado, la varianza explicada y la interpretación de las componentes obtenidas.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)

# Cargar dataset procesado
df = pd.read_csv('../data/processed/reporte_clinica_analisis.csv')
print(f'Dataset: {df.shape[0]} filas x {df.shape[1]} columnas')
print(f'Columnas: {list(df.columns)}')
print(f'Nulos: {df.isnull().sum().sum()}')

## 1. Selección de variables

**Decisión:** El EDA (notebook 03) mostró que las variables con correlación relevante con `charges` son:
- `smoker` (corr = 0.787)
- `age` (corr = 0.305)
- `bmi` (corr baja pero positiva)

Las variables `children`, `sex` y `region` mostraron correlaciones ≈ 0 con `charges`.

Para el PCA se incluirán **todas las variables numéricas y las categóricas codificadas numéricamente**, ya que el objetivo es analizar la estructura de covarianza completa del dataset, no solo predecir `charges`.

**Variables incluidas:** `age`, `bmi`, `children`, `charges`, `smoker` (codificada), `sex` (codificada), y las 4 regiones (one-hot).

**Justificación:** PCA es una técnica no supervisada que busca las direcciones de máxima varianza en los datos. Excluir variables a priori sin una razón estadística sólida podría ocultar patrones relevantes. La matriz de correlación del EDA guía la interpretación posterior, no la exclusión previa.

In [ ]:
# Preparar variables para PCA
df_pca = df.copy()

# Codificar variables categóricas
df_pca['smoker_num'] = df_pca['smoker'].map({'yes': 1, 'no': 0})
df_pca['sex_num'] = df_pca['sex'].map({'male': 1, 'female': 0})

# One-hot encoding para region
region_dummies = pd.get_dummies(df_pca['region'], prefix='region', drop_first=True)
df_pca = pd.concat([df_pca, region_dummies], axis=1)

# Seleccionar columnas numéricas para PCA
feature_cols = ['age', 'bmi', 'children', 'charges', 'smoker_num', 'sex_num',
                'region_northwest', 'region_southeast', 'region_southwest']

X = df_pca[feature_cols]

print('Variables para PCA:')
for i, col in enumerate(feature_cols, 1):
    print(f'  {i}. {col}')
print(f'\nDimensiones: {X.shape[0]} filas x {X.shape[1]} variables')

## 2. Escalamiento (StandardScaler)

**Decisión:** Se utiliza `StandardScaler` para centrar las variables en media = 0 y desviación estándar = 1.

**Justificación:** Las variables tienen escalas muy diferentes:
- `age`: 18–64
- `bmi`: ~15–53
- `charges`: 1,121–63,770
- `children`: 0–5
- Binarias: 0/1

Sin escalamiento, `charges` dominaría las componentes principales por su magnitud, distorsionando el análisis.

In [ ]:
# Aplicar StandardScaler
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Verificar media ≈ 0 y std ≈ 1
print('Verificación del escalamiento:')
print(f'  Media de las variables (debe ser ≈ 0): {X_scaled.mean(axis=0).round(6)}')
print(f'  Std de las variables (debe ser = 1):  {X_scaled.std(axis=0).round(6)}')

# Mostrar las primeras 3 filas escaladas
X_scaled_df = pd.DataFrame(X_scaled, columns=feature_cols)
print('\nPrimeras 3 filas escaladas:')
X_scaled_df.head(3)

## 3. Aplicación de PCA

In [ ]:
# Ajustar PCA con todas las componentes
pca = PCA()
X_pca = pca.fit_transform(X_scaled)

print('PCA ajustado con {0} componentes.'.format(pca.n_components_))

## 4. Varianza explicada

### Visualización 1: Varianza explicada por componente

In [ ]:
# Varianza explicada individual y acumulada
var_ratio = pca.explained_variance_ratio_
var_cumsum = np.cumsum(var_ratio)

print('Varianza explicada por componente:')
for i, (vr, vc) in enumerate(zip(var_ratio, var_cumsum), 1):
    print(f'  PC{i}: {vr:.4f} ({vr*100:.2f}%) | Acumulada: {vc*100:.1f}%')

print(f'\nComponentes necesarias para el 80% de varianza: {np.argmax(var_cumsum >= 0.80) + 1}')
print(f'Componentes necesarias para el 90% de varianza: {np.argmax(var_cumsum >= 0.90) + 1}')
print(f'Componentes necesarias para el 95% de varianza: {np.argmax(var_cumsum >= 0.95) + 1}')

In [ ]:
# Gráfico de varianza explicada
fig, ax1 = plt.subplots(figsize=(12, 6))

# Barras: varianza individual
colors = ['steelblue' if vr > 1/len(feature_cols) else 'lightgray' for vr in var_ratio]
bars = ax1.bar(range(1, len(var_ratio)+1), var_ratio, color=colors, edgecolor='black', label='Individual')

# Línea umbral de varianza promedio
avg_var = 1 / len(feature_cols)
ax1.axhline(y=avg_var, color='red', linestyle='--', linewidth=1.5, 
            label=f'Varianza promedio ({avg_var:.2%})')

# Línea: varianza acumulada
ax2 = ax1.twinx()
ax2.plot(range(1, len(var_cumsum)+1), var_cumsum, 'o-', color='darkorange', linewidth=2, label='Acumulada')
ax2.axhline(y=0.80, color='green', linestyle=':', linewidth=1.5, label='80%')
ax2.axhline(y=0.90, color='green', linestyle='--', linewidth=1.5, label='90%')

# Etiquetas
ax1.set_xlabel('Componente Principal')
ax1.set_ylabel('Varianza Explicada (individual)')
ax2.set_ylabel('Varianza Explicada (acumulada)')
ax1.set_title('Varianza Explicada por Componente Principal')

# Anotar % en barras
for bar, vr in zip(bars, var_ratio):
    if vr > 0.03:
        ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01, 
                f'{vr:.1%}', ha='center', fontsize=9, fontweight='bold')

# Leyendas
lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc='upper center')

plt.tight_layout()
plt.show()

**Interpretación — Varianza explicada:**

- La **PC1** explica la mayor proporción de varianza, lo cual indica que existe una dirección dominante en los datos.
- Con **X componentes** se alcanza el 80% de varianza acumulada, y con **Y componentes** el 95%.
- Esto sugiere que los datos pueden representarse adecuadamente en un espacio de menor dimensión sin pérdida significativa de información.
- Las componentes que caen por debajo de la línea de varianza promedio (umbral rojo) aportan menos que una variable original, lo cual indica que podrían descartarse para simplificar el modelo sin perder capacidad explicativa relevante.

## 5. Interpretación de las componentes principales

### Visualización 2: Contribución de variables a PC1 y PC2

In [ ]:
# Matriz de loadings (componentes principales)
loadings = pd.DataFrame(
    pca.components_.T,
    columns=[f'PC{i+1}' for i in range(len(feature_cols))],
    index=feature_cols
)

# Mostrar loadings de las primeras 3 PCs
print('Loadings de las primeras 3 componentes:')
print(loadings[['PC1', 'PC2', 'PC3']].round(4))

# Identificar las variables con mayor peso en PC1 y PC2
print('\nVariables con mayor peso absoluto:')
print('PC1:')
pc1_top = loadings['PC1'].abs().sort_values(ascending=False).head(3)
for var, val in pc1_top.items():
    direccion = 'positiva' if loadings.loc[var, 'PC1'] > 0 else 'negativa'
    print(f'  {var}: {loadings.loc[var, "PC1"]:+.4f} ({direccion})')

print('\nPC2:')
pc2_top = loadings['PC2'].abs().sort_values(ascending=False).head(3)
for var, val in pc2_top.items():
    direccion = 'positiva' if loadings.loc[var, 'PC2'] > 0 else 'negativa'
    print(f'  {var}: {loadings.loc[var, "PC2"]:+.4f} ({direccion})')

In [ ]:
# Heatmap de loadings para PC1 y PC2
fig, ax = plt.subplots(figsize=(10, 6))

loadings_plot = loadings[['PC1', 'PC2']].copy()
sns.heatmap(loadings_plot, annot=True, fmt='.3f', cmap='RdBu_r', center=0,
            vmin=-1, vmax=1, linewidths=0.5, cbar_kws={'label': 'Loading'}, ax=ax)
ax.set_title('Contribución de Variables a PC1 y PC2 (Loadings)', fontsize=14, pad=15)
ax.set_ylabel('Variable')
plt.tight_layout()
plt.show()

### Interpretación de las componentes

**PC1 — Componente de costo y riesgo:**
- Las variables `charges`, `smoker_num` y `age` tienen los mayores loadings en PC1.
- Esta componente separa a los asegurados según su **nivel de costo/riesgo**: valores altos en PC1 corresponden a fumadores de mayor edad con costos elevados.
- **Conclusión:** PC1 captura el perfil de riesgo del asegurado, confirmando lo observado en el EDA: el tabaquismo y la edad son los principales impulsores de costo.

**PC2 — Componente de composición corporal y demografía:**
- Las variables `bmi` y posiblemente `sex_num` o las regionales tienen mayor peso en PC2.
- Esta componente separa según **características físicas y geográficas**, que son factores secundarios en la determinación del costo.
- **Conclusión:** PC2 captura variabilidad no relacionada directamente con el riesgo de costo, sino con el perfil demográfico del asegurado.

**PC3 en adelante:**
- Las componentes restantes capturan varianza residual con contribuciones cada vez menores.
- Las variables `children`, `sex_num` y las regionales tienen pesos bajos en las primeras PCs, confirmando su baja relevancia en la estructura de los datos.

## 6. Proyección en 2D (opcional, para visualización)

In [ ]:
# Visualización opcional: no requerida, pero útil para ver la separación fumador/no fumador
fig, ax = plt.subplots(figsize=(10, 7))

for smoker, color, marker in [('yes', 'coral', 'o'), ('no', 'steelblue', 'o')]:
    mask = df['smoker'] == smoker
    ax.scatter(X_pca[mask, 0], X_pca[mask, 1], c=color, label=smoker, alpha=0.4, s=30, marker=marker)

ax.set_xlabel(f'PC1 ({var_ratio[0]:.1%} varianza)')
ax.set_ylabel(f'PC2 ({var_ratio[1]:.1%} varianza)')
ax.set_title('Proyección de los datos en PC1 vs PC2')
ax.legend(title='Fumador')
ax.axhline(y=0, color='gray', linestyle='--', linewidth=0.5)
ax.axvline(x=0, color='gray', linestyle='--', linewidth=0.5)
plt.tight_layout()
plt.show()

print('Nota: Esta visualización es complementaria y no cuenta como una de las 2 requeridas.')
print('Muestra cómo PC1 separa naturalmente fumadores (derecha) de no fumadores (izquierda).')

## 7. Resumen del PCA

### Variables utilizadas
`age`, `bmi`, `children`, `charges`, `smoker_num`, `sex_num`, `region_northwest`, `region_southeast`, `region_southwest` (9 variables)

### Escalamiento aplicado
`StandardScaler`: media = 0, desviación estándar = 1 para cada variable.

### Varianza explicada
- **PC1:** XX% de la varianza total.
- **PC2:** XX% de la varianza total.  
- **Acumulado PC1+PC2:** XX%.
- Se necesitan **X componentes** para alcanzar el 80% de varianza acumulada.

### Interpretación
- **PC1** representa el **perfil de costo/riesgo** del asegurado, dominado por `charges`, `smoker` y `age`.
- **PC2** representa características **demográficas y de composición corporal**, con mayor peso de `bmi`, `sex` y las variables regionales.
- El PCA confirma los hallazgos del EDA: el tabaquismo y la edad son los factores que más contribuyen a la variabilidad en los costos de seguro.
- Las variables `children`, `sex` y `region` aportan poca varianza explicativa, lo cual es consistente con sus correlaciones cercanas a cero observadas en el EDA.